In [80]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command



llm = init_chat_model("openai:gpt-4o-mini")
conn = sqlite3.connect(
    "memory.db",
    check_same_thread = False,
)

config = {
    "configurable":{
        "thread_id":"1",
    },
}




In [81]:
class State(MessagesState):
    custom_stuff: str

graph_builder = StateGraph(State)



In [82]:
# @tool
# def get_weather(city: str):
#     """Gets weather in city"""

#     return f"The weather in {city} is sunny."

# llm_with_tools = llm.bind_tools(
#     tools = [
#         get_weather,
#     ]
# )

@tool
def get_human_feedback(poem: str):
    """Asks the user for feedback on teh pome.
        Use this before returning the final respons.
    """ 
    feedback = interrupt(f"here is the poem, tell me what you think\n{poem}")
    return feedback['feedback']




llm_with_tools = llm.bind_tools(
    tools = [
        get_human_feedback,
    ]
)

def chatbot(state:State):
    # response = llm_with_tools.invoke(state["messages"])
    response = llm_with_tools.invoke( 
        f"""
        You are an expert in making poems.
        Use the `get_human_feedback` tool to get feedback on your poem.
        Only after you receive positive feedback you can return the final poem.
        ALWAYS ASK FOR FEEDBACK FIRST.
        Here is the conversation history :
        {state["messages"]},
        """
    )
    return {
        "messages": [response],
    }


In [83]:
tool_node = ToolNode(
    tools = [
        get_human_feedback,
    ],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools",tool_node)

graph_builder.add_edge(START,"chatbot")
graph_builder.add_conditional_edges("chatbot",tools_condition)
graph_builder.add_edge("tools","chatbot")

graph = graph_builder.compile(
    checkpointer = SqliteSaver(conn),
    )



In [84]:
result = graph.invoke(
    {
        "messages":[
            {"role" : "user", "content":"please make a poem about Python code."},
        ]
    },
    config = config,
)

In [85]:
for message in result["messages"]:
    message.pretty_print()
    

================================ Human Message =================================

please make a poem about Python code.
================================ Human Message =================================

please make a poem about Python code.
================================ Human Message =================================

please make a poem about Python code.
================================== Ai Message ==================================

In a world of logic, sleek and bright,  
Python dances with code, a pure delight.  
Indentations guide, like whispers in the night,  
Simplicity in syntax, everything feels right.  

Variables flow like rivers so wide,  
Lists and dictionaries, side by side.  
Loops that iterate, in joy they glide,  
With a touch of elegance, no place to hide.  

Functions wrapped in a silky embrace,  
Reusable magic, a programmer's grace.  
Imports and modules, each with its place,  
Building in layers, a structured space.  

Errors might stumble, a bug may appear,  


In [86]:
snapshot = graph.get_state(config)

snapshot.interrupts

(Interrupt(value="here is the poem, tell me what you think\nIn a world of logic, sleek and bright,  \nPython dances with code, a pure delight.  \nIndentations guide, like whispers in the night,  \nSimplicity in syntax, everything feels right.  \n\nVariables flow like rivers so wide,  \nLists and dictionaries, side by side.  \nLoops that iterate, in joy they glide,  \nWith a touch of elegance, no place to hide.  \n\nFunctions wrapped in a silky embrace,  \nReusable magic, a programmer's grace.  \nImports and modules, each with its place,  \nBuilding in layers, a structured space.  \n\nErrors might stumble, a bug may appear,  \nBut with a few tests, the solution is near.  \nA community vibrant, as friends we cheer,  \nIn Python's realm, we conquer our fear.  \n\nFrom data analysis to web's vast expanse,  \nMachine learning dreams take a daring chance.  \nWith every line penned, a new romance,  \nIn Python's embrace, we all find our dance.  \n\nSo here’s to the language, versatile and bol

In [ ]:
response = Command(
    resume={"feedback": "It looks good!"},
    
)

result = graph.invoke(
    response,
    config = config,
)